# MR V1: Band-Based Mean Reversion

**Intraday TWAP mean reversion** with rolling std bands and ADX regime filter.

| Feature | Detail |
|---------|--------|
| Entry | Price < lower band (long) or > upper band (short) |
| Exit | Price crosses back through TWAP or SL hit |
| Regime | ADX < 30 (ranging market only) |
| EOD | Forced exit at 21:00 UTC |
| Eval freq | Every 5 minutes |
| Leverage | Fixed 1x |

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import vectorbtpro as vbt
from numba import njit
from numba.core.errors import NumbaPerformanceWarning
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)
warnings.filterwarnings(
    "ignore",
    message="Argument at index .* not found in SequenceTaker",
    module=r"vectorbtpro\\.utils\\.chunking",
)

In [ ]:
import multiprocessing
from numba import get_num_threads
from pathlib import Path

n_cores = multiprocessing.cpu_count()
print(f"Cores: {n_cores} | Numba threads: {get_num_threads()}")

def _fullscreen(fig):
    fig.update_layout(width=None, height=None, autosize=True,
        margin=dict(l=30, r=30, t=60, b=30),
        title=dict(font=dict(size=20), x=0.5, xanchor="center"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
    return fig

vbt.settings.set("plotting.pre_show_func", _fullscreen)
vbt.settings.returns.year_freq = pd.Timedelta(hours=24) * 252

## 2. Data

In [ ]:
_PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path(".").resolve()
# When run from notebooks/ dir: parent is project root
# When run from project root: "." is project root
# Fallback: search upward for data/
for _p in [Path(".").resolve(), Path(".").resolve().parent, Path(".").resolve().parent.parent]:
    if (_p / "data" / "EUR-USD.parquet").exists():
        _PROJECT_ROOT = _p
        break

def load_fx_data(path="data/EUR-USD.parquet", shift_hours=0):
    resolved = _PROJECT_ROOT / path
    data_raw = vbt.Data.from_parquet(str(resolved))
    symbol = data_raw.symbols[0]
    df = data_raw.data[symbol].set_index("date").sort_index()
    if shift_hours:
        df.index = df.index + pd.Timedelta(hours=shift_hours)
    raw = df.copy()
    raw.columns = [c.lower() for c in raw.columns]
    df.columns = [c.capitalize() for c in df.columns]
    data = vbt.Data.from_data({symbol: df}, tz_localize=False, tz_convert=False)
    return raw, data

raw, data = load_fx_data()
index_ns = vbt.dt.to_ns(raw.index)

# Quick stats
bars_per_day = raw.index.to_series().groupby(raw.index.date).count()
ann_factor = 252.0 * bars_per_day.mean()
print(f"Bars: {len(raw):,} | Range: {raw.index[0]} -> {raw.index[-1]}")
print(f"Ann factor: {ann_factor:.0f}")

In [ ]:
# Raw data inspection
print("=== OHLC Data ===")
print(f"Shape: {raw.shape}")
print(f"Columns: {raw.columns.tolist()}")
print(f"Dtypes:\n{raw.dtypes}")
print(f"\nNaN counts:\n{raw.isna().sum()}")
raw.head(10)

In [ ]:
raw.describe()

In [ ]:
# Data overview
fig = go.Figure(data=go.Scatter(x=raw.index, y=raw["close"], line=dict(color="#333", width=1)))
fig.update_layout(height=400, title="EUR/USD Close Price Overview")
fig.show()

In [ ]:
# Daily returns distribution
daily_close = raw["close"].resample("1D").last().dropna()
daily_rets = daily_close.pct_change().dropna()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Daily Returns Distribution", "QQ-like: Sorted Returns"))
fig.add_trace(go.Histogram(x=daily_rets.values, nbinsx=100, name="Daily Returns",
                           marker_color="#2196F3"), row=1, col=1)
sorted_rets = np.sort(daily_rets.values)
fig.add_trace(go.Scatter(y=sorted_rets, mode="lines", name="Sorted Returns",
                         line=dict(color="#FF5722")), row=1, col=2)
fig.update_layout(height=350, showlegend=False)
fig.show()

## 3. Numba Kernels

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# NUMBA KERNELS (all @njit(nogil=True) for parallel execution)
# ═══════════════════════════════════════════════════════════════════════

@njit(nogil=True)
def find_day_boundaries_nb(index_ns):
    n = len(index_ns)
    start_idx = np.empty(n, dtype=np.int64)
    end_idx = np.empty(n, dtype=np.int64)
    if n == 0:
        return start_idx, end_idx, 0
    day_number = vbt.dt_nb.days_nb(ts=index_ns)
    current_day = day_number[0]
    day_counter = 0
    current_start = 0
    for i in range(1, n):
        if day_number[i] != current_day:
            start_idx[day_counter] = current_start
            end_idx[day_counter] = i
            day_counter += 1
            current_day = day_number[i]
            current_start = i
    start_idx[day_counter] = current_start
    end_idx[day_counter] = n
    day_counter += 1
    return start_idx, end_idx, day_counter


@njit(nogil=True)
def compute_adx_nb(high, low, close, period):
    n = len(close)
    adx = np.full(n, np.nan)
    if n < period + 1:
        return adx
    plus_dm = np.empty(n); minus_dm = np.empty(n); tr = np.empty(n)
    plus_dm[0] = 0.0; minus_dm[0] = 0.0; tr[0] = high[0] - low[0]
    for i in range(1, n):
        up = high[i] - high[i-1]; down = low[i-1] - low[i]
        plus_dm[i] = up if (up > down and up > 0) else 0.0
        minus_dm[i] = down if (down > up and down > 0) else 0.0
        hl = high[i] - low[i]; hc = abs(high[i] - close[i-1]); lc = abs(low[i] - close[i-1])
        tr[i] = max(hl, max(hc, lc))
    s_plus = vbt.generic.nb.ewm_mean_1d_nb(plus_dm, span=period, minp=period, adjust=False)
    s_minus = vbt.generic.nb.ewm_mean_1d_nb(minus_dm, span=period, minp=period, adjust=False)
    s_tr = vbt.generic.nb.ewm_mean_1d_nb(tr, span=period, minp=period, adjust=False)
    dx = np.full(n, np.nan)
    for i in range(n):
        if not np.isnan(s_tr[i]) and s_tr[i] > 1e-10:
            pdi = s_plus[i] / s_tr[i]; mdi = s_minus[i] / s_tr[i]
            s = pdi + mdi
            if s > 1e-10:
                dx[i] = abs(pdi - mdi) / s * 100.0
    adx[:] = vbt.generic.nb.ewm_mean_1d_nb(dx, span=period, minp=period, adjust=False)
    return adx


@njit(nogil=True)
def compute_daily_adx_broadcast_nb(index_ns, high, low, close, open_, adx_period):
    n = len(close)
    start_arr, end_arr, n_days = find_day_boundaries_nb(index_ns)
    if n_days < adx_period + 2:
        return np.full(n, np.nan)
    d_high = np.empty(n_days); d_low = np.empty(n_days); d_close = np.empty(n_days)
    for d in range(n_days):
        s, e = start_arr[d], end_arr[d]
        mx, mn = high[s], low[s]
        for i in range(s+1, e):
            if high[i] > mx: mx = high[i]
            if low[i] < mn: mn = low[i]
        d_high[d] = mx; d_low[d] = mn; d_close[d] = close[e-1]
    daily_adx = compute_adx_nb(d_high, d_low, d_close, adx_period)
    adx_minute = np.full(n, np.nan)
    for d in range(1, n_days):
        val = daily_adx[d-1]
        for i in range(start_arr[d], end_arr[d]):
            adx_minute[i] = val
    return adx_minute


@njit(nogil=True)
def compute_adx_regime_nb(index_ns, high, low, close, open_, adx_period, adx_threshold):
    n = len(close)
    adx = compute_daily_adx_broadcast_nb(index_ns, high, low, close, open_, adx_period)
    regime_ok = np.ones(n, dtype=np.float64)
    for i in range(n):
        if not np.isnan(adx[i]) and adx[i] > adx_threshold:
            regime_ok[i] = 0.0
    return regime_ok


@njit(nogil=True)
def compute_intraday_twap_nb(index_ns, high, low, close):
    n = len(close)
    twap = np.full(n, np.nan)
    if n == 0:
        return twap
    start_arr, end_arr, n_days = find_day_boundaries_nb(index_ns)
    for d in range(n_days):
        s, e = start_arr[d], end_arr[d]
        cum_tp = 0.0; count = 0
        for i in range(s, e):
            tp = (high[i] + low[i] + close[i]) / 3.0
            if not np.isnan(tp):
                cum_tp += tp; count += 1
                twap[i] = cum_tp / count
    return twap


@njit(nogil=True)
def compute_intraday_rolling_std_nb(index_ns, data, lookback):
    n = len(data)
    out = np.full(n, np.nan)
    if n == 0:
        return out
    start_arr, end_arr, n_days = find_day_boundaries_nb(index_ns)
    minp = min(lookback, 20)
    for d in range(n_days):
        s, e = start_arr[d], end_arr[d]
        if e - s < minp:
            continue
        day_std = vbt.generic.nb.rolling_std_1d_nb(data[s:e], lookback, minp=minp, ddof=1)
        for i in range(e - s):
            out[s + i] = day_std[i]
    return out


@njit(nogil=True)
def compute_intraday_zscore_nb(index_ns, data, lookback):
    n = len(data)
    out = np.full(n, np.nan)
    if n == 0:
        return out
    start_arr, end_arr, n_days = find_day_boundaries_nb(index_ns)
    minp = min(lookback, 20)
    for d in range(n_days):
        s, e = start_arr[d], end_arr[d]
        if e - s < minp:
            continue
        day_z = vbt.generic.nb.rolling_zscore_1d_nb(data[s:e], lookback, minp=minp, ddof=1)
        for i in range(e - s):
            out[s + i] = day_z[i]
    return out


@njit(nogil=True)
def compute_deviation_nb(close, twap):
    n = len(close)
    dev = np.empty(n)
    for i in range(n):
        if np.isnan(twap[i]) or np.isnan(close[i]):
            dev[i] = np.nan
        else:
            dev[i] = close[i] - twap[i]
    return dev


@njit(nogil=True)
def compute_intraday_bands_nb(index_ns, twap, deviation, lookback, band_width):
    n = len(twap)
    rstd = compute_intraday_rolling_std_nb(index_ns, deviation, lookback)
    upper = np.full(n, np.nan); lower = np.full(n, np.nan)
    for i in range(n):
        s = rstd[i]
        if not np.isnan(s) and s > 1e-10 and not np.isnan(twap[i]):
            upper[i] = twap[i] + band_width * s
            lower[i] = twap[i] - band_width * s
    return upper, lower


@njit(nogil=True)
def compute_mr_base_indicators_nb(index_ns, high, low, close, open_,
                                   lookback, band_width, adx_period, adx_threshold):
    twap = compute_intraday_twap_nb(index_ns, high, low, close)
    deviation = compute_deviation_nb(close, twap)
    zscore = compute_intraday_zscore_nb(index_ns, deviation, lookback)
    upper, lower = compute_intraday_bands_nb(index_ns, twap, deviation, lookback, band_width)
    regime_ok = compute_adx_regime_nb(index_ns, high, low, close, open_, adx_period, adx_threshold)
    return twap, zscore, upper, lower, regime_ok


@njit(nogil=True)
def mr_band_signal_nb(c, close_arr, upper_arr, lower_arr, twap_arr,
                       regime_ok_arr, index_ns_arr, eod_hour_arr, eod_minute_arr, eval_freq_arr):
    ts_ns = index_ns_arr[c.i]
    cur_h = vbt.dt_nb.hour_nb(ts_ns); cur_m = vbt.dt_nb.minute_nb(ts_ns)
    eod_h = vbt.pf_nb.select_nb(c, eod_hour_arr); eod_m = vbt.pf_nb.select_nb(c, eod_minute_arr)
    if (cur_h > eod_h) or (cur_h == eod_h and cur_m >= eod_m):
        el = vbt.pf_nb.ctx_helpers.in_long_position_nb(c)
        es = vbt.pf_nb.ctx_helpers.in_short_position_nb(c)
        return False, el, False, es
    ef = vbt.pf_nb.select_nb(c, eval_freq_arr)
    if ef > 0 and cur_m % ef != 0:
        return False, False, False, False
    px = vbt.pf_nb.select_nb(c, close_arr)
    ub = vbt.pf_nb.select_nb(c, upper_arr); lb = vbt.pf_nb.select_nb(c, lower_arr)
    tw = vbt.pf_nb.select_nb(c, twap_arr); regime = vbt.pf_nb.select_nb(c, regime_ok_arr)
    if np.isnan(px) or np.isnan(ub) or np.isnan(lb) or np.isnan(tw):
        return False, False, False, False
    il = vbt.pf_nb.ctx_helpers.in_long_position_nb(c)
    ish = vbt.pf_nb.ctx_helpers.in_short_position_nb(c)
    if not il and not ish:
        if regime < 0.5: return False, False, False, False
        if px < lb: return True, False, False, False
        elif px > ub: return False, False, True, False
    elif il:
        if px >= tw: return False, True, False, False
    elif ish:
        if px <= tw: return False, False, False, True
    return False, False, False, False

## 4. Indicators

In [ ]:
MR_V1 = vbt.IF(
    class_name="MR_V1", short_name="mr_v1",
    input_names=["index_ns", "high_minute", "low_minute", "close_minute", "open_minute"],
    param_names=["lookback", "band_width", "adx_period", "adx_threshold"],
    output_names=["twap", "zscore", "upper_band", "lower_band", "regime_ok"],
).with_apply_func(
    compute_mr_base_indicators_nb, takes_1d=True,
    lookback=60, band_width=2.0, adx_period=14, adx_threshold=30.0)

ind = MR_V1.run(
    index_ns, raw["high"], raw["low"], raw["close"], raw["open"],
    lookback=60, band_width=2.0, adx_period=14, adx_threshold=30.0,
    jitted_loop=True, jitted_warmup=True)
print("Outputs:", ind.output_names)

### Indicator Inspection

In [ ]:
# Build indicator DataFrame for inspection
ind_df = pd.DataFrame({
    "close": raw["close"].values,
    "twap": ind.twap.values,
    "zscore": ind.zscore.values,
    "upper_band": ind.upper_band.values,
    "lower_band": ind.lower_band.values,
    "regime_ok": ind.regime_ok.values,
}, index=raw.index)

print("=== Indicators DataFrame ===")
print(f"Shape: {ind_df.shape}")
print(f"\nNaN counts:\n{ind_df.isna().sum()}")
print(f"\nFirst valid rows (after warmup):")
ind_df.dropna().head(10)

In [ ]:
# Indicator statistics
ind_df.describe()

## 5. Signal Visualization

In [ ]:
n = min(7200, len(raw)); sl = slice(-n, None)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
    subplot_titles=("Price + TWAP + Bands", "Z-Score", "ADX Regime Filter"))

fig.add_trace(go.Scatter(x=raw.index[sl], y=raw["close"].values[sl], name="Close",
    line=dict(color="#333", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.twap.values[sl], name="TWAP",
    line=dict(color="#FF9800", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.upper_band.values[sl], name="Upper Band",
    line=dict(color="#E91E63", dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.lower_band.values[sl], name="Lower Band",
    line=dict(color="#E91E63", dash="dot")), row=1, col=1)

fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.zscore.values[sl], name="Z-Score",
    line=dict(color="#2196F3")), row=2, col=1)
fig.add_hline(y=2, line_dash="dot", line_color="red", row=2, col=1)
fig.add_hline(y=-2, line_dash="dot", line_color="green", row=2, col=1)
fig.add_hline(y=0, line_color="gray", row=2, col=1)

fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.regime_ok.values[sl], name="Regime OK",
    fill="tozeroy", line=dict(color="rgba(76,175,80,0.5)")), row=3, col=1)

fig.update_layout(height=900, title="MR V1 Indicators")
fig.show()

## 6. Backtest

In [ ]:
pf = vbt.Portfolio.from_signals(
    close=raw["close"], open=raw["open"], high=raw["high"], low=raw["low"],
    signal_func_nb=mr_band_signal_nb,
    signal_args=(
        vbt.Rep("close_arr"), vbt.Rep("upper_arr"), vbt.Rep("lower_arr"),
        vbt.Rep("twap_arr"), vbt.Rep("regime_ok_arr"), vbt.Rep("index_ns_arr"),
        vbt.Rep("eod_hour_arr"), vbt.Rep("eod_minute_arr"), vbt.Rep("eval_freq_arr")),
    broadcast_named_args=dict(
        close_arr=raw["close"], upper_arr=ind.upper_band.values,
        lower_arr=ind.lower_band.values, twap_arr=ind.twap.values,
        regime_ok_arr=ind.regime_ok.values, index_ns_arr=index_ns,
        eod_hour_arr=21, eod_minute_arr=0, eval_freq_arr=5),
    leverage=1.0, slippage=0.00015, sl_stop=0.005,
    init_cash=1_000_000, freq="1min")
print(f"Trades: {pf.trades.count()}")

## 7. Results

### Orders & Trades Inspection

In [ ]:
# Orders log
print(f"=== Orders: {pf.orders.count()} ===")
orders_df = pf.orders.records_readable
print(orders_df.head(20).to_string())

In [ ]:
# Trades log
print(f"=== Trades: {pf.trades.count()} ===")
if pf.trades.count() > 0:
    trades_df = pf.trades.records_readable
    print(trades_df.head(20).to_string())
    print(f"\n=== Trade PnL Distribution ===")
    pnl = trades_df["PnL"].dropna()
    print(f"  Mean:   {pnl.mean():.4f}")
    print(f"  Median: {pnl.median():.4f}")
    print(f"  Std:    {pnl.std():.4f}")
    print(f"  Min:    {pnl.min():.4f}")
    print(f"  Max:    {pnl.max():.4f}")
    print(f"  Win%:   {(pnl > 0).mean()*100:.1f}%")
else:
    print("No trades generated.")

In [ ]:
# Position summary: time in market
print(f"=== Position Coverage ===")
pos = pf.position_mask
in_market_pct = pos.sum() / len(pos) * 100
print(f"  In market: {pos.sum():,} bars ({in_market_pct:.1f}%)")
print(f"  Flat:      {(~pos).sum():,} bars ({100-in_market_pct:.1f}%)")

### Portfolio Stats

In [ ]:
print("PORTFOLIO STATS")
print("=" * 50)
print(pf.stats().to_string())
if pf.trades.count() > 0:
    print(f"\nTRADE STATS\n{'='*50}")
    print(pf.trades.stats().to_string())

### Equity Curve & Drawdowns

In [ ]:
# Resample to daily for fast plotting (minute data has 3M+ points)
pf_daily = pf.resample("1D")
fig = pf_daily.plot(subplots=["cumulative_returns", "drawdowns", "underwater", "trade_pnl"])
fig.update_layout(height=900, title="Portfolio Summary (daily)")
fig.show()

### Trade Signals on Price

In [ ]:
n_bars = min(7200, len(raw))
sim_start = raw.index[-n_bars]
fig = pf.plot_trade_signals(plot_positions="zones", sim_start=sim_start)
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.twap.values[-n_bars:],
    name="TWAP", line=dict(color="#FF9800", dash="dash")))
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.upper_band.values[-n_bars:],
    name="Upper", line=dict(color="#E91E63", dash="dot")))
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.lower_band.values[-n_bars:],
    name="Lower", line=dict(color="#E91E63", dash="dot")))
fig.update_layout(height=600, title="Trade Signals (last week)")
fig.show()

### Trade Analysis

In [ ]:
if pf.trades.count() > 0:
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=("Trade PnL (%)", "MAE", "MFE", "Running Edge Ratio"))
    pf.trades.plot_pnl(pct_scale=True, fig=fig, add_trace_kwargs=dict(row=1, col=1))
    pf.trades.plot_mae(fig=fig, add_trace_kwargs=dict(row=1, col=2))
    pf.trades.plot_mfe(fig=fig, add_trace_kwargs=dict(row=2, col=1))
    pf.trades.plot_running_edge_ratio(fig=fig, add_trace_kwargs=dict(row=2, col=2))
    fig.update_layout(height=800, showlegend=False)
    fig.show()

### Monthly Returns Heatmap

In [ ]:
rets = pf_daily.returns
monthly = rets.resample("ME").apply(lambda x: (1 + x).prod() - 1)
df_m = pd.DataFrame({"return": monthly})
df_m["year"] = df_m.index.year; df_m["month"] = df_m.index.month
pivot = df_m.pivot_table(values="return", index="year", columns="month", aggfunc="first")
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot.columns = month_names[:len(pivot.columns)]

fig = go.Figure(data=go.Heatmap(
    z=pivot.values * 100, x=pivot.columns.tolist(), y=[str(y) for y in pivot.index],
    colorscale="RdYlGn", texttemplate="%{z:.1f}%", textfont=dict(size=10), zmid=0))
fig.update_layout(title="Monthly Returns (%)", height=300 + len(pivot) * 30)
fig.show()

### Rolling Sharpe Ratio (Daily)

In [ ]:
# Use daily-resampled portfolio for fast rolling calc
daily_rets = pf_daily.returns
rolling_sharpe = daily_rets.rolling(252).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values,
                         name="Rolling Sharpe (1Y)", line=dict(color="#2196F3")))
fig.add_hline(y=0, line_dash="solid", line_color="gray")
fig.add_hline(y=1, line_dash="dot", line_color="green", annotation_text="Sharpe=1")
fig.update_layout(height=350, title="Rolling 1-Year Sharpe Ratio (Daily)")
fig.show()

## Parameter Sweep

In [ ]:
ind_sweep = MR_V1.run(
    index_ns, raw["high"], raw["low"], raw["close"], raw["open"],
    lookback=vbt.Param([20, 40, 60, 120, 240]),
    band_width=vbt.Param([1.5, 2.0, 2.5, 3.0]),
    adx_period=14, adx_threshold=30.0,
    jitted_loop=True, jitted_warmup=True, param_product=True)
print(f"Param combos: {ind_sweep.wrapper.shape_2d[1]}")

In [ ]:
pf_sweep = vbt.Portfolio.from_signals(
    close=raw["close"], open=raw["open"], high=raw["high"], low=raw["low"],
    signal_func_nb=mr_band_signal_nb,
    signal_args=(
        vbt.Rep("close_arr"), vbt.Rep("upper_arr"), vbt.Rep("lower_arr"),
        vbt.Rep("twap_arr"), vbt.Rep("regime_ok_arr"), vbt.Rep("index_ns_arr"),
        vbt.Rep("eod_hour_arr"), vbt.Rep("eod_minute_arr"), vbt.Rep("eval_freq_arr")),
    broadcast_named_args=dict(
        close_arr=raw["close"], upper_arr=ind_sweep.upper_band,
        lower_arr=ind_sweep.lower_band, twap_arr=ind_sweep.twap,
        regime_ok_arr=ind_sweep.regime_ok, index_ns_arr=index_ns,
        eod_hour_arr=21, eod_minute_arr=0, eval_freq_arr=5),
    leverage=1.0, slippage=0.00015, sl_stop=0.005,
    init_cash=1_000_000, freq="1min",
    jitted=dict(parallel=True), chunked="threadpool")
print("Sweep done.")

### Sweep Rankings

In [25]:
# Parameter ranking table
sweep_summary = pd.DataFrame({
    "Sharpe": pf_sweep.sharpe_ratio,
    "Total Return": pf_sweep.total_return,
    "Max DD": pf_sweep.max_drawdown,
    "Trades": pf_sweep.trades.count(),
    "Win Rate": pf_sweep.trades.win_rate,
}).sort_values("Sharpe", ascending=False)

print("=== Top 10 Parameter Combos (by Sharpe) ===")
print(sweep_summary.head(10).to_string())
print(f"\n=== Bottom 5 ===")
print(sweep_summary.tail(5).to_string())

=== Top 10 Parameter Combos (by Sharpe) ===
                                                                        Sharpe  Total Return    Max DD  Trades  Win Rate
mr_v1_lookback mr_v1_band_width mr_v1_adx_period mr_v1_adx_threshold                                                    
20             1.5              14               30.0                -6.805194     -0.934157 -0.934168    8973  0.432966
               2.0              14               30.0                -6.805194     -0.934157 -0.934168    8973  0.432966
               2.5              14               30.0                -6.805194     -0.934157 -0.934168    8973  0.432966
               3.0              14               30.0                -6.805194     -0.934157 -0.934168    8973  0.432966
40             1.5              14               30.0                -6.805194     -0.934157 -0.934168    8973  0.432966
               2.0              14               30.0                -6.805194     -0.934157 -0.934168    897

### Sweep Heatmaps

In [26]:
fig = pf_sweep.sharpe_ratio.vbt.heatmap(x_level="mr_v1_lookback", y_level="mr_v1_band_width")
fig.update_layout(title="Sharpe Ratio Heatmap")
fig.show()

In [27]:
fig = pf_sweep.total_return.vbt.heatmap(x_level="mr_v1_lookback", y_level="mr_v1_band_width")
fig.update_layout(title="Total Return Heatmap")
fig.show()

In [28]:
fig = pf_sweep.max_drawdown.vbt.heatmap(x_level="mr_v1_lookback", y_level="mr_v1_band_width")
fig.update_layout(title="Max Drawdown Heatmap")
fig.show()

In [29]:
fig = pf_sweep.trades.count().vbt.heatmap(x_level="mr_v1_lookback", y_level="mr_v1_band_width")
fig.update_layout(title="Trade Count Heatmap")
fig.show()

## Cross-Validation (Walk-Forward)

Using VBT Pro native `@vbt.cv_split` decorator.

In [30]:
# 80/20 holdout split
split_idx = int(len(raw) * 0.8)
raw_train = raw.iloc[:split_idx].copy()
raw_test = raw.iloc[split_idx:].copy()
print(f"Train: {len(raw_train):,} bars ({raw_train.index[0]} -> {raw_train.index[-1]})")
print(f"Test:  {len(raw_test):,} bars ({raw_test.index[0]} -> {raw_test.index[-1]})")

Train: 2,444,604 bars (2018-01-01 19:01:00 -> 2024-08-02 06:23:00)
Test:  611,152 bars (2024-08-02 06:24:00 -> 2026-04-01 00:00:00)


In [31]:
# Define CV pipeline with @vbt.cv_split
def selection_func(grid_results):
    return vbt.LabelSel([grid_results.idxmax()])

@vbt.cv_split(
    splitter="from_purged_walkforward",
    splitter_kwargs=dict(n_folds=10, n_test_folds=1, min_train_folds=3, purge_td="1 hour"),
    takeable_args=["raw_data"],
    merge_func="concat",
    selection=vbt.RepFunc(selection_func),
    return_grid="all",
)
def mr_v1_cv(raw_data, lookback, band_width):
    ns = vbt.dt.to_ns(raw_data.index)
    ind_cv = MR_V1.run(ns, raw_data["high"], raw_data["low"], raw_data["close"], raw_data["open"],
        lookback=lookback, band_width=band_width, adx_period=14, adx_threshold=30.0,
        jitted_loop=True, jitted_warmup=True)
    pf_cv = vbt.Portfolio.from_signals(
        close=raw_data["close"], open=raw_data["open"], high=raw_data["high"], low=raw_data["low"],
        signal_func_nb=mr_band_signal_nb,
        signal_args=(
            vbt.Rep("close_arr"), vbt.Rep("upper_arr"), vbt.Rep("lower_arr"),
            vbt.Rep("twap_arr"), vbt.Rep("regime_ok_arr"), vbt.Rep("index_ns_arr"),
            vbt.Rep("eod_hour_arr"), vbt.Rep("eod_minute_arr"), vbt.Rep("eval_freq_arr")),
        broadcast_named_args=dict(
            close_arr=raw_data["close"], upper_arr=ind_cv.upper_band.values,
            lower_arr=ind_cv.lower_band.values, twap_arr=ind_cv.twap.values,
            regime_ok_arr=ind_cv.regime_ok.values, index_ns_arr=ns,
            eod_hour_arr=21, eod_minute_arr=0, eval_freq_arr=5),
        leverage=1.0, slippage=0.00015, sl_stop=0.005,
        init_cash=1_000_000, freq="1min")
    return pf_cv.sharpe_ratio

In [ ]:
# Run CV
grid_results, best_results = mr_v1_cv(
    raw_train,
    vbt.Param([20, 60, 120, 240]),
    vbt.Param([1.5, 2.0, 2.5, 3.0]),
)

 14%|#4        | 1/7 [00:18<01:50, 18.35s/it, split=1]

In [ ]:
# Display CV results
print("Grid results (all splits x params):")
print(grid_results)
print(f"\nBest results per split:")
print(best_results)

## Holdout Test

In [ ]:
# TODO: Extract best params from CV and rerun on train/test
# For now, display the CV selection results
print("CV analysis complete.")
print("Best params selected per split are shown above.")
print("Next step: validate on holdout set with selected params.")